## 🎯 Learning Objectives
* Understand the importance and mechanisms of structured output for LLMs in agentic workflows.
* Learn how to define and enforce structured output using Pydantic models and LangChain's capabilities.
* Explore techniques for guiding agents towards specific tool usage and ensuring structured inputs for those tools.
* Identify common use cases and performance considerations for structured output and forced tool use in production agent systems.


## Structured Output Agents and Forced Tool Use

In the realm of Agentic AI, Large Language Models (LLMs) are the brains, but for them to be truly effective in complex workflows, they often need to do more than just generate free-form text. They need to interact with external systems, update databases, or trigger specific actions. This is where **structured output** and **forced tool use** become indispensable.

### The Challenge: Bridging Text and Structure

Imagine asking an LLM to "Extract the customer's name, email, and order ID from this support ticket." Without guidance, the LLM might return a conversational summary, a bulleted list, or even just the name. While human-readable, this unstructured output is difficult for a downstream system (like a CRM or an order fulfillment API) to parse reliably. This is akin to a chef being asked to "make dinner" and returning a delicious meal, but without a recipe or a clear list of ingredients used – great for eating, bad for replicating or scaling.

### Structured Output: The Recipe for LLMs

**Structured output** refers to compelling an LLM to generate responses that conform to a predefined schema, typically JSON. This schema acts like a strict recipe, dictating the exact format, data types, and fields the LLM must adhere to. LangChain, leveraging modern LLM capabilities like OpenAI's Function Calling or Google Gemini's Function Calling, makes this incredibly straightforward. By providing the LLM with a schema (often defined using Python's `Pydantic` library), we can ensure its output is consistently parseable and ready for programmatic use.

**Why is it crucial?**
*   **Reliability:** Eliminates the need for brittle regex or complex NLP parsers on LLM output.
*   **Integration:** Directly consumable by APIs, databases, and other software components.
*   **Consistency:** Ensures uniform data across multiple LLM invocations.
*   **Reduced Hallucination:** By constraining the output, the LLM is less likely to invent non-existent fields or values.

### Forced Tool Use: Guiding the Agent's Actions

Agents are designed to be autonomous, choosing the best tool for a given task. However, there are scenarios where we need to ensure an agent *must* use a particular tool, or at least provide its inputs in a specific, structured manner. This is **forced tool use**.

Consider an agent tasked with "Calculate the square root of 144." While an LLM might be able to *estimate* it, a calculator tool will provide the precise answer. "Forcing" tool use here means ensuring the agent invokes the calculator tool with the correct, structured input (`number=144`, `operation='sqrt'`) rather than attempting to compute it itself or hallucinating a result.

**Techniques for Forced Tool Use:**
1.  **Explicit Prompting:** Crafting prompts that strongly suggest or demand the use of a specific tool for certain types of queries.
2.  **Tool Input Schemas:** Defining the expected input for a tool using Pydantic, so the LLM *must* generate structured arguments when calling that tool.
3.  **Agent Type Selection:** Using agent executors (like `create_tool_calling_agent` in LangChain) that are inherently optimized and designed to leverage LLM function calling for tool invocation.
4.  **`with_structured_output` for Tool Inputs:** Applying structured output techniques not just to the final response, but also to the *arguments* an LLM generates for a tool call.

By combining structured output for general responses and structured input for tool calls, we gain fine-grained control over our agents, making them more predictable, reliable, and capable of integrating seamlessly into complex automated systems.


In [ ]:
# Ensure you have the necessary libraries installed:
# pip install langchain langchain-openai pydantic

import os
from typing import List, Optional

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent

# --- Configuration --- 
# Set your OpenAI API key. In a production environment, use environment variables or a secure secret management system.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Initialize the LLM. We'll use a powerful model capable of function calling.
# As of 2026, gpt-4o is a strong contender for general agentic tasks.
llm = ChatOpenAI(model="gpt-4o", temperature=0)

print("--- Demonstrating Structured Output (LLM Response) ---")

# 1. Define a Pydantic model for the desired structured output
class ContactInfo(BaseModel):
    name: str = Field(description="The full name of the person.")
    email: Optional[str] = Field(description="The email address of the person, if available.")
    phone: Optional[str] = Field(description="The phone number of the person, if available.")
    company: Optional[str] = Field(description="The company the person works for, if available.")

# 2. Create a chain that forces the LLM to return output conforming to the Pydantic model
# LangChain's `with_structured_output` is the modern way to do this.
structured_llm = llm.with_structured_output(ContactInfo)

# 3. Invoke the structured LLM with a prompt
structured_output_prompt = "Extract contact information from the following text: 'John Doe, CEO of ExampleCorp, can be reached at john.doe@examplecorp.com or 555-123-4567.'"
print(f"Prompt: {structured_output_prompt}")

# The output will be an instance of the ContactInfo Pydantic model
contact_data = structured_llm.invoke(structured_output_prompt)
print(f"Structured Output Type: {type(contact_data)}")
print(f"Structured Output: {contact_data.json(indent=2)}")

# Accessing fields directly
print(f"Extracted Name: {contact_data.name}")
print(f"Extracted Email: {contact_data.email}")

print("\n--- Demonstrating Forced Tool Use (Structured Tool Input) ---")

# 1. Define a tool with a Pydantic model for its input arguments
class CalculatorInput(BaseModel):
    expression: str = Field(description="The mathematical expression to evaluate, e.g., '2 + 2' or 'sqrt(144)'.")

@tool("calculator", args_schema=CalculatorInput)
def calculate(expression: str) -> str:
    """Evaluates a mathematical expression and returns the result."""
    try:
        # Using eval is generally unsafe for untrusted input, but for demonstration
        # purposes with controlled LLM output, it's illustrative.
        # In a real system, use a safer math evaluation library.
        result = str(eval(expression))
        return f"Result of '{expression}': {result}"
    except Exception as e:
        return f"Error evaluating expression '{expression}': {e}"

# 2. Define a list of tools available to the agent
tools = [calculate]

# 3. Create a prompt template for the agent
# The `tools` and `tool_names` variables are automatically populated by create_tool_calling_agent
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. You must use the calculator tool for any mathematical questions."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

# 4. Create a tool-calling agent
# This agent type is designed to leverage LLM function calling capabilities for tool use.
agent = create_tool_calling_agent(llm, tools, agent_prompt)

# 5. Create an AgentExecutor to run the agent
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 6. Invoke the agent with a mathematical question
forced_tool_use_prompt = "What is the square root of 144 multiplied by 3?"
print(f"Prompt: {forced_tool_use_prompt}")

# Observe the agent's thought process and tool invocation
agent_executor.invoke({"input": forced_tool_use_prompt})

print("\n--- Demonstrating another forced tool use scenario ---")
forced_tool_use_prompt_2 = "Calculate 15% of 200."
print(f"Prompt: {forced_tool_use_prompt_2}")
agent_executor.invoke({"input": forced_tool_use_prompt_2})


### Interpreting the Output and Performance Trade-offs

#### Structured Output Interpretation

In the first part of the code, you'll observe that the `structured_llm.invoke()` call returns an instance of our `ContactInfo` Pydantic model, not just a string. This means the LLM has successfully parsed the input text and mapped the relevant information to the `name`, `email`, `phone`, and `company` fields as defined in our schema. The `.json(indent=2)` method then pretty-prints this Pydantic object as a JSON string, demonstrating its direct usability for programmatic access.

This output is highly reliable. If the LLM cannot find a specific piece of information (e.g., a phone number), that field will be `None` (or its default value), rather than the LLM hallucinating or returning an empty string in an inconsistent format. This consistency is invaluable for downstream processing.

#### Forced Tool Use Interpretation

In the second part, when the agent is prompted with a mathematical question, you'll see a `tool_code` block in the verbose output. This indicates that the LLM, guided by the system prompt and its inherent function-calling capabilities, decided to use the `calculator` tool. Crucially, the arguments passed to the `calculator` tool (`expression='sqrt(144) * 3'`) are automatically generated by the LLM and conform to the `CalculatorInput` Pydantic schema we defined for the tool. This demonstrates:

1.  **Agent Selection:** The agent correctly identified that a mathematical tool was needed.
2.  **Structured Input:** The agent provided the tool's arguments in the exact structured format (a single `expression` string) required by the `CalculatorInput` schema.

This is a powerful form of "forced" tool use: the agent is not just *allowed* to use tools, but is *guided* to use them for specific tasks, and its interaction with those tools is strictly governed by predefined schemas. This prevents the agent from attempting to perform calculations itself (which LLMs are notoriously bad at) or providing malformed inputs to the tool.

#### Performance Trade-offs

**Advantages:**
*   **Increased Reliability:** Drastically reduces parsing errors and ensures data integrity, leading to more robust applications.
*   **Simplified Downstream Logic:** Eliminates complex post-processing of LLM outputs, making integration with other systems much easier.
*   **Enhanced Control:** Gives developers precise control over the LLM's output format and tool interaction, making agents more predictable.
*   **Reduced Hallucination:** By constraining the LLM's response space, it's less likely to invent data or misuse tools.

**Disadvantages:**
*   **Increased Token Usage (Potentially):** Complex Pydantic schemas can add to the prompt length, increasing token consumption and latency, especially with larger models.
*   **Strictness:** Overly strict schemas might prevent the LLM from expressing nuanced information if it doesn't fit the predefined structure. Careful schema design is key.
*   **LLM Capability Dependence:** Relies on LLMs with strong function-calling or structured output capabilities (e.g., OpenAI's `gpt-4o`, Google's `gemini-pro`). Older or less capable models may struggle to adhere to complex schemas.

#### Typical Use Cases

*   **Data Extraction:** Extracting entities (names, dates, amounts) from unstructured text like emails, documents, or customer reviews.
*   **API Call Generation:** Generating structured JSON payloads for external API calls (e.g., booking systems, CRM updates, payment gateways).
*   **Database Query Generation:** Creating structured queries (e.g., SQL, NoSQL) based on natural language requests.
*   **Automated Form Filling:** Populating web forms or application fields with extracted information.
*   **Workflow Orchestration:** Ensuring specific steps in a multi-agent workflow receive and produce data in a consistent, machine-readable format.
*   **Code Generation:** Generating code snippets that adhere to specific function signatures or class structures.


### Resources

*   **LangChain Structured Output:** [https://python.langchain.com/docs/how_to/structured_output/](https://python.langchain.com/docs/how_to/structured_output/)
*   **LangChain Tools:** [https://python.langchain.com/docs/modules/agents/tools/](https://python.langchain.com/docs/modules/agents/tools/)
*   **LangChain Agents:** [https://python.langchain.com/docs/modules/agents/](https://python.langchain.com/docs/modules/agents/)
*   **Pydantic Documentation:** [https://docs.pydantic.dev/latest/](https://docs.pydantic.dev/latest/)
*   **OpenAI Function Calling:** [https://platform.openai.com/docs/guides/function-calling](https://platform.openai.com/docs/guides/function-calling)
*   **Google Gemini Function Calling:** [https://ai.google.dev/docs/function_calling](https://ai.google.dev/docs/function_calling)
